# Pakistan Stock Market Historical Data Pipeline

## Project Overview

This project collects historical stock-price data for major companies listed on the Pakistan Stock Exchange (PSX).

The data is downloaded from Yahoo Finance using Python, cleaned with Pandas, and combined into a single master dataset.

## Data Period

**January 2015 – December 2025**

## Project Objectives

The prepared dataset will be used for:

* Return and risk analysis
* Correlation analysis
* Portfolio optimization
* Efficient frontier construction
* Portfolio backtesting
* Interactive dashboard development

## Data Pipeline

```text
PSX Stock Symbols
        ↓
Yahoo Finance
        ↓
Python and Pandas
        ↓
Data Cleaning
        ↓
Master PSX Dataset
        ↓
Portfolio Optimization
```


In [19]:
# Install required Python libraries

!pip install pandas yfinance openpyxl --quiet

In [ ]:
# Import required libraries

import pandas as pd
import yfinance as yf
from pathlib import Path
import time

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# ============================================================
# PROJECT FOLDER SETUP
# ============================================================

PROJECT_ROOT = Path(
    r"C:\Users\MOHSIN\OneDrive\Desktop\Pakistan-Stock-Market-Analytics"
)

# Define project folder locations

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

SAMPLE_DATA_DIR = PROJECT_ROOT / "data" / "sample"

LOGS_DIR = PROJECT_ROOT / "logs"


# Create required folders if they do not exist

for folder in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    SAMPLE_DATA_DIR,
    LOGS_DIR
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )


print("Project setup successful!")

print("\nProject folder:")
print(PROJECT_ROOT)

print("\nProcessed data folder:")
print(PROCESSED_DATA_DIR)

print("\nLogs folder:")
print(LOGS_DIR)

Project setup successful!

Project folder:
C:\Users\MOHSIN\OneDrive\Desktop\Pakistan-Stock-Market-Analytics

Processed data folder:
C:\Users\MOHSIN\OneDrive\Desktop\Pakistan-Stock-Market-Analytics\data\processed

Logs folder:
C:\Users\MOHSIN\OneDrive\Desktop\Pakistan-Stock-Market-Analytics\logs


In [ ]:
# ============================================================
# MAJOR PSX COMPANY SYMBOLS
# ============================================================

psx_symbols = [

    # Commercial Banks

    "MEBL", "HBL", "UBL", "MCB", "ABL",
    "BAHL", "BAFL", "NBP", "BOP", "AKBL",
    "FABL", "JSBL", "SNBL", "BIPL",


    # Oil and Gas

    "OGDC", "PPL", "POL", "MARI",
    "PSO", "APL", "SHEL", "HASCOL",
    "ATRL", "NRL", "PRL",


    # Cement

    "LUCK", "DGKC", "MLCF", "FCCL",
    "KOHC", "CHCC", "PIOC", "ACPL",
    "POWER", "THCCL",


    # Fertilizer

    "FFC", "EFERT", "FATIMA",


    # Technology and Communication

    "SYS", "TRG", "NETSOL", "AVN",
    "AIRLINK", "OCTOPUS", "PTC", "TELE",


    # Power Generation

    "HUBC", "KAPCO", "NCPL",
    "NPL", "EPQL", "KOHE",


    # Automobile

    "INDU", "HCAR", "PSMC",
    "MTL", "AGTL", "GHNI",
    "GHNL", "SAZEW", "ATLH",


    # Pharmaceuticals

    "SEARL", "AGP", "GLAXO",
    "HINOON", "FEROZ", "ABOT",
    "CPHL",


    # Food and Consumer

    "NESTLE", "COLG", "UPFL",
    "UNITY", "NATF", "PREMA",


    # Chemicals

    "EPCL", "LOTCHEM", "ICI",
    "DOL",


    # Textile

    "ILP", "NML", "NCL",
    "KTML", "GATM", "ANL",


    # Engineering and Steel

    "ISL", "MUGHAL", "ASTL",
    "ASL", "INIL", "AGHA",


    # Insurance

    "EFUG", "JLICL", "AICL",
    "IGIHL",


    # Holdings and Other Companies

    "ENGROH", "DAWH", "PAKT",
    "PABC", "THALL", "IBFL"
]


# Remove duplicate symbols

psx_symbols = list(
    dict.fromkeys(
        psx_symbols
    )
)


print(
    "Total unique PSX symbols:",
    len(psx_symbols)
)

Total unique PSX symbols: 100


In [ ]:
# ============================================================
# PSX HISTORICAL DATA PIPELINE
# Yahoo Finance
# Period: 2015 to 2025
# ============================================================


# Store successfully downloaded company data temporarily

all_data = []


# Store failed company information

failed_symbols = []


print("=" * 65)

print("PSX DATA DOWNLOAD STARTED")

print("=" * 65)


# Download data for each PSX symbol one by one

for number, symbol in enumerate(
    psx_symbols,
    start=1
):


    # Create the Yahoo Finance ticker

    yahoo_ticker = symbol + ".KA"


    print(
        f"\n{number}/{len(psx_symbols)} "
        f"Downloading {yahoo_ticker}..."
    )


    try:


        # Download historical data from Yahoo Finance

        df = yf.download(

            yahoo_ticker,

            start="2015-01-01",

            end="2026-01-01",

            auto_adjust=False,

            progress=False,

            threads=False

        )


        # Handle cases where Yahoo Finance returns no data

        if df.empty:


            print(
                "No data available ❌"
            )


            failed_symbols.append(

                {

                    "Symbol": symbol,

                    "Yahoo_Ticker":
                    yahoo_ticker,

                    "Reason":
                    "No data returned"

                }

            )


            continue


        # Normalize Yahoo Finance's multi-level columns

        if isinstance(

            df.columns,

            pd.MultiIndex

        ):


            df.columns = (

                df.columns

                .get_level_values(0)

            )


        # Reset the date index to a column

        df = df.reset_index()


        # Add the company symbol

        df["Symbol"] = symbol


        # Keep only the required columns

        df = df[

            [

                "Date",

                "Symbol",

                "Open",

                "High",

                "Low",
                "Close",

                "Adj Close",

                "Volume"

            ]

        ]


        # Convert the date column to a proper datetime format

        df["Date"] = pd.to_datetime(

            df["Date"]

        )


        # Remove duplicate date and symbol rows

        df = df.drop_duplicates(

            subset=[

                "Date",

                "Symbol"

            ]

        )


        # Remove rows with missing closing prices

        df = df.dropna(

            subset=[

                "Close"

            ]

        )


        # Sort by date

        df = df.sort_values(

            "Date"

        )


        # Save the raw CSV file for each company

        company_file = (

            RAW_DATA_DIR

            / f"{symbol}_historical_data.csv"

        )


        df.to_csv(

            company_file,

            index=False

        )


        # Add the cleaned data to the master list

        all_data.append(

            df

        )


        print(

            f"Success ✅ | "

            f"Rows: {len(df)} | "

            f"From: {df['Date'].min().date()} | "

            f"To: {df['Date'].max().date()}"

        )


        # Add a short delay between requests

        time.sleep(0.5)


    except Exception as error:


        print(

            f"Failed ❌ | {error}"

        )


        failed_symbols.append(

            {

                "Symbol": symbol,

                "Yahoo_Ticker":
                yahoo_ticker,

                "Reason":
                str(error)

            }

        )


# ============================================================
# COMBINE SUCCESSFUL COMPANY DATA
# ============================================================


if len(all_data) > 0:


    master_data = pd.concat(

        all_data,

        ignore_index=True

    )


    # Sort by symbol and date

    master_data = (

        master_data

        .sort_values(

            [

                "Symbol",

                "Date"

            ]

        )

        .reset_index(

            drop=True

        )

    )


    # Define the master CSV location

    master_file = (

        PROCESSED_DATA_DIR

        / "PSX_2015_2025_Master_Data.csv"

    )


    # Save the master data

    master_data.to_csv(

        master_file,

        index=False

    )


    # Create a small sample dataset for GitHub

    sample_data = (

        master_data

        .groupby(

            "Symbol",

            group_keys=False

        )

        .head(10)

    )


    sample_file = (

        SAMPLE_DATA_DIR

        / "PSX_Sample_Data.csv"

    )


    sample_data.to_csv(

        sample_file,

        index=False

    )


    print(

        "\n"

        + "=" * 65

    )


    print(

        "MASTER DATASET CREATED SUCCESSFULLY ✅"

    )


    print(

        "=" * 65

    )


    print(

        "Successful companies:",

        master_data[

            "Symbol"

        ].nunique()

    )


    print(

        "Total rows:",

        len(master_data)

    )


    print(

        "Master file:"

    )


    print(

        master_file

    )


else:


    print(

        "\nNo company data was downloaded."

    )


# ============================================================
# SAVE FAILED SYMBOL REPORT
# ============================================================


failed_report = pd.DataFrame(

    failed_symbols

)


failed_file = (

    LOGS_DIR

    / "failed_symbols.csv"

)


failed_report.to_csv(

    failed_file,

    index=False

)


print(

    "\nFailed companies:",

    len(failed_symbols)

)


print(

    "Failed-symbol report:"

)


print(

    failed_file

)

PSX DATA DOWNLOAD STARTED

1/100 Downloading MEBL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

2/100 Downloading HBL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

3/100 Downloading UBL.KA...
Success ✅ | Rows: 2852 | From: 2015-01-01 | To: 2025-12-31

4/100 Downloading MCB.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

5/100 Downloading ABL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

6/100 Downloading BAHL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

7/100 Downloading BAFL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

8/100 Downloading NBP.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

9/100 Downloading BOP.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

10/100 Downloading AKBL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

11/100 Downloading FABL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

$ENGROH.KA: possibly delisted; no timezone found

1 Failed download:
['ENGROH.KA']: possibly delisted; no timezone found


No data available ❌

96/100 Downloading DAWH.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

97/100 Downloading PAKT.KA...
Success ✅ | Rows: 2854 | From: 2015-01-01 | To: 2025-12-31

98/100 Downloading PABC.KA...
Success ✅ | Rows: 1143 | From: 2021-07-19 | To: 2025-12-31

99/100 Downloading THALL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

100/100 Downloading IBFL.KA...
Success ✅ | Rows: 2853 | From: 2015-01-01 | To: 2025-12-31

MASTER DATASET CREATED SUCCESSFULLY ✅
Successful companies: 99
Total rows: 269825
Master file:
C:\Users\MOHSIN\OneDrive\Desktop\Pakistan-Stock-Market-Analytics\data\processed\PSX_2015_2025_Master_Data.csv

Failed companies: 1
Failed-symbol report:
C:\Users\MOHSIN\OneDrive\Desktop\Pakistan-Stock-Market-Analytics\logs\failed_symbols.csv


In [ ]:
# ============================================================
# PSX HISTORICAL DATA PIPELINE
# Yahoo Finance
# Period: 2015 to 2025
# ============================================================


# Store successfully downloaded company data temporarily

all_data = []


# Store failed company information

failed_symbols = []


print("=" * 65)

print("PSX DATA DOWNLOAD STARTED")

print("=" * 65)


# Download data for each PSX symbol one by one

for number, symbol in enumerate(
    psx_symbols,
    start=1
):


    # Create the Yahoo Finance ticker

    yahoo_ticker = symbol + ".KA"


    print(
        f"\n{number}/{len(psx_symbols)} "
        f"Downloading {yahoo_ticker}..."
    )


    try:


        # Download historical data from Yahoo Finance

        df = yf.download(

            yahoo_ticker,

            start="2015-01-01",

            end="2026-01-01",

            auto_adjust=False,

            progress=False,

            threads=False

        )


        # Handle cases where Yahoo Finance returns no data

        if df.empty:


            print(
                "No data available ❌"
            )


            failed_symbols.append(

                {

                    "Symbol": symbol,

                    "Yahoo_Ticker":
                    yahoo_ticker,

                    "Reason":
                    "No data returned"

                }

            )


            continue


        # Normalize Yahoo Finance's multi-level columns

        if isinstance(

            df.columns,

            pd.MultiIndex

        ):


            df.columns = (

                df.columns

                .get_level_values(0)

            )


        # Reset the date index to a column

        df = df.reset_index()


        # Add the company symbol

        df["Symbol"] = symbol


        # Keep only the required columns

        df = df[

            [

                "Date",

                "Symbol",

                "Open",

                "High",

                "Low",
                "Close",

                "Adj Close",

                "Volume"

            ]

        ]


        # Convert the date column to a proper datetime format

        df["Date"] = pd.to_datetime(

            df["Date"]

        )


        # Remove duplicate date and symbol rows

        df = df.drop_duplicates(

            subset=[

                "Date",

                "Symbol"

            ]

        )


        # Remove rows with missing closing prices

        df = df.dropna(

            subset=[

                "Close"

            ]

        )


        # Sort by date

        df = df.sort_values(

            "Date"

        )


        # Save the raw CSV file for each company

        company_file = (

            RAW_DATA_DIR

            / f"{symbol}_historical_data.csv"

        )


        df.to_csv(

            company_file,

            index=False

        )


        # Add the cleaned data to the master list

        all_data.append(

            df

        )


        print(

            f"Success ✅ | "

            f"Rows: {len(df)} | "

            f"From: {df['Date'].min().date()} | "

            f"To: {df['Date'].max().date()}"

        )


        # Add a short delay between requests

        time.sleep(0.5)


    except Exception as error:


        print(

            f"Failed ❌ | {error}"

        )


        failed_symbols.append(

            {

                "Symbol": symbol,

                "Yahoo_Ticker":
                yahoo_ticker,

                "Reason":
                str(error)

            }

        )


# ============================================================
# COMBINE SUCCESSFUL COMPANY DATA
# ============================================================


if len(all_data) > 0:


    master_data = pd.concat(

        all_data,

        ignore_index=True

    )


    # Sort by symbol and date

    master_data = (

        master_data

        .sort_values(

            [

                "Symbol",

                "Date"

            ]

        )

        .reset_index(

            drop=True

        )

    )


    # Define the master CSV location

    master_file = (

        PROCESSED_DATA_DIR

        / "PSX_2015_2025_Master_Data.csv"

    )


    # Save the master data

    master_data.to_csv(

        master_file,

        index=False

    )


    # Create a small sample dataset for GitHub

    sample_data = (

        master_data

        .groupby(

            "Symbol",

            group_keys=False

        )

        .head(10)

    )


    sample_file = (

        SAMPLE_DATA_DIR

        / "PSX_Sample_Data.csv"

    )


    sample_data.to_csv(

        sample_file,

        index=False

    )


    print(

        "\n"

        + "=" * 65

    )


    print(

        "MASTER DATASET CREATED SUCCESSFULLY ✅"

    )


    print(

        "=" * 65

    )


    print(

        "Successful companies:",

        master_data[

            "Symbol"

        ].nunique()

    )


    print(

        "Total rows:",

        len(master_data)

    )


    print(

        "Master file:"

    )


    print(

        master_file

    )


else:


    print(

        "\nNo company data was downloaded."

    )


# ============================================================
# SAVE FAILED SYMBOL REPORT
# ============================================================


failed_report = pd.DataFrame(

    failed_symbols

)


failed_file = (

    LOGS_DIR

    / "failed_symbols.csv"

)


failed_report.to_csv(

    failed_file,

    index=False

)


print(

    "\nFailed companies:",

    len(failed_symbols)

)


print(

    "Failed-symbol report:"

)


print(

    failed_file

)

PSX DATA DOWNLOAD STARTED

1/100 Downloading MEBL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

2/100 Downloading HBL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

3/100 Downloading UBL.KA...
Success ✅ | Rows: 2852 | From: 2015-01-01 | To: 2025-12-31

4/100 Downloading MCB.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

5/100 Downloading ABL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

6/100 Downloading BAHL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

7/100 Downloading BAFL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

8/100 Downloading NBP.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

9/100 Downloading BOP.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

10/100 Downloading AKBL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

11/100 Downloading FABL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

$ENGROH.KA: possibly delisted; no timezone found

1 Failed download:
['ENGROH.KA']: possibly delisted; no timezone found


No data available ❌

96/100 Downloading DAWH.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

97/100 Downloading PAKT.KA...
Success ✅ | Rows: 2854 | From: 2015-01-01 | To: 2025-12-31

98/100 Downloading PABC.KA...
Success ✅ | Rows: 1143 | From: 2021-07-19 | To: 2025-12-31

99/100 Downloading THALL.KA...
Success ✅ | Rows: 2851 | From: 2015-01-01 | To: 2025-12-31

100/100 Downloading IBFL.KA...
Success ✅ | Rows: 2853 | From: 2015-01-01 | To: 2025-12-31

MASTER DATASET CREATED SUCCESSFULLY ✅
Successful companies: 99
Total rows: 269825
Master file:
C:\Users\MOHSIN\OneDrive\Desktop\Pakistan-Stock-Market-Analytics\data\processed\PSX_2015_2025_Master_Data.csv

Failed companies: 1
Failed-symbol report:
C:\Users\MOHSIN\OneDrive\Desktop\Pakistan-Stock-Market-Analytics\logs\failed_symbols.csv


# Step 2: Data Validation and Quality Assessment

This section validates the combined PSX historical dataset before financial analysis and portfolio optimization.

The following checks are performed:

- Dataset dimensions
- Number of unique stocks
- Historical date coverage
- Missing values
- Duplicate observations
- Company-level data availability

In [ ]:
# ============================================================
# LOAD THE MASTER PSX DATASET
# ============================================================

import pandas as pd
from pathlib import Path


# Define the project root

PROJECT_ROOT = Path(
    r"C:\Users\MOHSIN\OneDrive\Desktop\Pakistan-Stock-Market-Analytics"
)


# Define the master dataset location

MASTER_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "PSX_2015_2025_Master_Data.csv"
)


# Load the CSV file into a DataFrame

stock_data = pd.read_csv(
    MASTER_FILE
)


# Convert the Date column to a proper datetime format

stock_data["Date"] = pd.to_datetime(
    stock_data["Date"]
)


print(
    "Master dataset loaded successfully! ✅"
)


# Display the first five rows

stock_data.head()

Master dataset loaded successfully! ✅


,Date,Symbol,Open,High,Low,Close,Adj Close,Volume
0,2015-01-01,ABL,113.250000,113.900002,112.510002,113.550003,33.938301,283800
1,2015-01-02,ABL,113.000000,115.000000,113.000000,114.029999,34.081764,508800
2,2015-01-05,ABL,114.150002,114.500000,113.599998,114.010002,34.075790,384200
3,2015-01-06,ABL,113.800003,114.750000,113.500000,113.669998,33.974163,458500
4,2015-01-07,ABL,113.550003,114.290001,113.099998,113.599998,33.953247,240400


In [10]:
# ============================================================
# DATASET OVERVIEW
# ============================================================

print("=" * 55)

print("PSX MASTER DATASET SUMMARY")

print("=" * 55)


print(
    "\nTotal rows:",
    len(stock_data)
)


print(
    "Total columns:",
    stock_data.shape[1]
)


print(
    "Total companies:",
    stock_data["Symbol"].nunique()
)


print(
    "Starting date:",
    stock_data["Date"].min().date()
)


print(
    "Ending date:",
    stock_data["Date"].max().date()
)


print(
    "Duplicate records:",
    stock_data.duplicated(
        subset=[
            "Date",
            "Symbol"
        ]
    ).sum()
)

PSX MASTER DATASET SUMMARY

Total rows: 269825
Total columns: 8
Total companies: 99
Starting date: 2015-01-01
Ending date: 2025-12-31
Duplicate records: 0


In [11]:
# ============================================================
# CHECK MISSING VALUES
# ============================================================

missing_values = (
    stock_data
    .isnull()
    .sum()
)


missing_report = pd.DataFrame({

    "Column":
    missing_values.index,

    "Missing_Values":
    missing_values.values

})


missing_report

,Column,Missing_Values
0,Date,0
1,Symbol,0
2,Open,0
3,High,0
4,Low,0
5,Close,0
6,Adj Close,0
7,Volume,0


In [12]:
# ============================================================
# COMPANY-LEVEL DATA COVERAGE
# ============================================================

company_coverage = (

    stock_data

    .groupby(
        "Symbol"
    )

    .agg(

        First_Date=(
            "Date",
            "min"
        ),

        Last_Date=(
            "Date",
            "max"
        ),

        Total_Records=(
            "Date",
            "count"
        )

    )

    .reset_index()

)


# Sort companies by available records

company_coverage = (

    company_coverage

    .sort_values(

        "Total_Records",

        ascending=False

    )

)


company_coverage

,Symbol,First_Date,Last_Date,Total_Records
15,AVN,2015-01-01,2025-12-31,2854
27,EFUG,2015-01-01,2025-12-31,2854
56,KTML,2015-01-01,2025-12-31,2854
69,NESTLE,2015-01-01,2025-12-31,2854
77,PAKT,2015-01-01,2025-12-31,2854
...,...,...,...,...
3,AGHA,2020-11-02,2025-12-31,1328
22,CPHL,2021-07-12,2025-12-31,1148
76,PABC,2021-07-19,2025-12-31,1143
7,AIRLINK,2021-09-22,2025-12-31,1096
